In [29]:
import re
from imports import *
from scipy.stats import gaussian_kde

In [2]:
env_vars = dotenv_values(dotenv_path="../.env")
with open(f'.{env_vars["data_folder_path"]}/{env_vars["split_fname"]}', "rb") as f:
    data = pickle.load(f)

train_data = data["train_data"]
test_data = data["test_data"]
full_kvasir_data = train_data + test_data

### Plotting the KDE 

In [3]:
def get_mask_area_ratio(data):
    msk_transform = transforms.Compose([
        transforms.Resize(ast.literal_eval(env_vars.get("mask_size")), transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
        ])

    data_with_area = []
    for idx, value in enumerate(data):
        mask = msk_transform(value[1])
        area_ratio = (mask == 1).sum().item() / mask.numel()
        data_with_area.append((value, area_ratio))

    sorted_data_with_area = sorted(data_with_area, key=lambda x: x[1])
    area_ratios = [item[1] for item in sorted_data_with_area]
    return area_ratios

In [4]:
def plot_train_vs_test_kde(train_data, test_data, savefig_name=None, bw=0.5):
    full_dataset = train_data + test_data
    train_ratios = get_mask_area_ratio(train_data)
    test_ratios  = get_mask_area_ratio(test_data)
    full_ratios  = get_mask_area_ratio(full_dataset)

    # KDE
    x_vals = np.linspace(0, 1, 1000)
    train_kde = gaussian_kde(train_ratios, bw_method=bw)
    test_kde  = gaussian_kde(test_ratios,  bw_method=bw)
    full_kde  = gaussian_kde(full_ratios,  bw_method=bw)
    train_y = train_kde(x_vals)
    test_y  = test_kde(x_vals)
    full_y  = full_kde(x_vals)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    ax.plot(x_vals, train_y, color="#E74C3C", linewidth=2, label="Train KDE")
    ax.plot(x_vals, test_y,  color="#27AE60", linewidth=2, label="Test KDE")
    ax.plot(x_vals, full_y,  color="black",   linewidth=2, label="Full Dataset KDE", linestyle="--", alpha=0.7)
    ax.set_xlabel("Mask Area Ratio", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title("Dataset Distribution Comparison (KDE)", fontsize=14, pad=10)

    # Styling
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, which="major", alpha=0.25)
    ax.minorticks_on()
    ax.grid(True, which="minor", linestyle=":", alpha=0.1)

    # Ticks
    ax.set_xlim(0, 1)
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.tick_params(axis='both', which='major', labelsize=10)

    ax.legend(frameon=False, fontsize=10, loc="upper right")
    fig.tight_layout()

    if savefig_name:
        outdir = f'.{env_vars["experiments_folder_path"]}/figs'
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [5]:
plot_train_vs_test_kde(train_data, test_data, savefig_name="distbn_shift")

### Plotting #sample vs area ratios

In [6]:
def plot_area_vs_sample(
    area_ratios,
    data_type,
    savefig_name=None,
    x_tick_size=20,
):
    area_ratios = np.asarray(area_ratios, dtype=float)
    n = len(area_ratios)
    x = np.arange(1, n + 1)

    fig, ax = plt.subplots(figsize=(11, 6), dpi=300)
    ax.plot(x, area_ratios, linewidth=2, solid_capstyle="round")
    ax.set_title(f"{data_type} Dataset — Area Ratio vs Sample", pad=12)
    ax.set_xlabel("# Sample")
    ax.set_ylabel("Area Ratio")

    #styling
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.minorticks_on()
    ax.grid(True, which="major", alpha=0.25)
    ax.grid(True, which="minor", alpha=0.10, linestyle=":")

    #ticks
    step = max(1, n // x_tick_size)
    xticks = np.arange(0, n + 1, step)
    if xticks[-1] != n:
        xticks = np.append(xticks, n)
    ax.set_xticks(xticks)
    ax.set_xlim(1, n)
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0.0, 1.01, 0.1))

    # median and IQR
    q25, q50, q75 = np.percentile(area_ratios, [25, 50, 75])
    ax.axhline(q50, linestyle="--", linewidth=1.2, alpha=0.8, label=f"Median = {q50:.2f}")
    ax.fill_between([1, n], [q25, q25], [q75, q75], alpha=0.08, label="IQR (25–75%)")

    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()

    if savefig_name:
        outdir = f'.{env_vars["experiments_folder_path"]}/figs'
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


In [7]:
full_dataset = train_data+test_data
full_ratios = get_mask_area_ratio(full_dataset)
plot_area_vs_sample(full_ratios, data_type="Full", savefig_name="area_vs_sample", x_tick_size=20)

### Plotting Dice score vs Area butterfly

In [ ]:
def get_area_vs_dice(data, model_name, model_config, ckpt_path, device):
    img_transform = transforms.Compose([
            transforms.Resize(ast.literal_eval(env_vars.get("image_size")), transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor()
            ])
    msk_transform = transforms.Compose([
        transforms.Resize(ast.literal_eval(env_vars.get("mask_size")), transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
        ])
    

    model = select_model(model_name=model_name, model_config=model_config)
    
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    area_vs_score_list = []

    for i in range(len(data)):
        image, mask = img_transform(data[i][0]).to(device), msk_transform(data[i][1]).to(device)
        area_ratio = (mask == 1).sum().item()  / mask.numel()

        with torch.no_grad():
            preds = model(image.unsqueeze(0))
            if isinstance(preds, (tuple, list)):
                preds = preds[0]
            preds = preds.squeeze(0)
            dice_score = calculate_dice_score(preds=preds, targets=mask, device=device, model_name=model_name)

        area_vs_score_list.append((area_ratio, dice_score.item()))

    return area_vs_score_list

def _butterfly_data(results, threshold, n_bins):
    mask_bins = np.linspace(0.0, 1.0, n_bins + 1)
    data_bins = [[] for _ in range(n_bins)]

    for area, score in results:
        for j in range(n_bins):
            if mask_bins[j] <= area < mask_bins[j + 1]:
                data_bins[j].append(score)
                break

    butterfly_data = [
        (
            sum(score < threshold for score in bin_scores),  # below
            sum(score > threshold for score in bin_scores)   # above
        )
        for bin_scores in data_bins
    ]

    below_counts = [x[0] for x in butterfly_data]
    above_counts = [x[1] for x in butterfly_data]

    return mask_bins, below_counts, above_counts

In [ ]:
def get_area_vs_dice(data, model_name, model_config, ckpt_path, device):
    img_transform = transforms.Compose([
            transforms.Resize(ast.literal_eval(env_vars.get("image_size")), transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor()
            ])
    msk_transform = transforms.Compose([
        transforms.Resize(ast.literal_eval(env_vars.get("mask_size")), transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
        ])
    

    model = select_model(model_name=model_name, model_config=model_config)
    
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    area_vs_score_list = []

    for i in range(len(data)):
        image, mask = img_transform(data[i][0]).to(device), msk_transform(data[i][1]).to(device)
        area_ratio = (mask == 1).sum().item()  / mask.numel()

        with torch.no_grad():
            preds = model(image.unsqueeze(0))
            if isinstance(preds, (tuple, list)):
                preds = preds[0]
            preds = preds.squeeze(0)
            dice_score = calculate_dice_score(preds=preds, targets=mask, device=device, model_name=model_name)

        area_vs_score_list.append((area_ratio, dice_score.item()))

    return area_vs_score_list

def _butterfly_data(results, threshold, n_bins):
    mask_bins = np.linspace(0.0, 1.0, n_bins + 1)
    data_bins = [[] for _ in range(n_bins)]

    for area, score in results:
        for j in range(n_bins):
            if mask_bins[j] <= area < mask_bins[j + 1]:
                data_bins[j].append(score)
                break

    butterfly_data = [
        (
            sum(score < threshold for score in bin_scores),  # below
            sum(score > threshold for score in bin_scores)   # above
        )
        for bin_scores in data_bins
    ]

    below_counts = [x[0] for x in butterfly_data]
    above_counts = [x[1] for x in butterfly_data]

    return mask_bins, below_counts, above_counts


def plot_butterfly_mask_vs_score(n_bins, threshold, pre_results, post_results, savefig_name):
    mask_bins_pre, pre_below_counts, pre_above_counts = _butterfly_data(pre_results, threshold, n_bins)
    mask_bins_post, post_below_counts, post_above_counts = _butterfly_data(post_results, threshold, n_bins)

    bar_width = 0.35
    index = np.arange(n_bins)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

    # LEFT: PRE
    ax = axes[0]
    ax.bar(index, -np.array(pre_below_counts), bar_width, color='blue', label=f'Below (< DSC={threshold})')
    ax.bar(index,  np.array(pre_above_counts), bar_width, color='red',  label=f'Above (> DSC={threshold})')
    ax.axhline(y=0, color='black', linestyle='--')
    ax.set_title(f'Pre-Oversample (t={threshold})')

    for i in range(n_bins):
        ax.text(i, -pre_below_counts[i] - 0.5, str(pre_below_counts[i]), ha='center', va='top', fontsize=8)
        ax.text(i,  pre_above_counts[i] + 0.5, str(pre_above_counts[i]), ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Mask Area Ratio Bins')
    ax.set_ylabel('Dice Score Counts')
    ax.set_xticks(index)
    ax.set_xticklabels([f'{mask_bins_pre[i]:.2f}-{mask_bins_pre[i+1]:.2f}' for i in range(n_bins)], rotation=45)

    # RIGHT: POST
    ax = axes[1]
    ax.bar(index, -np.array(post_below_counts), bar_width, color='blue', label=f'Below (< DSC={threshold})')
    ax.bar(index,  np.array(post_above_counts), bar_width, color='red',  label=f'Above (> DSC={threshold})')
    ax.axhline(y=0, color='black', linestyle='--')
    ax.set_title(f'Post-Oversample (t={threshold})')

    for i in range(n_bins):
        ax.text(i, -post_below_counts[i] - 0.5, str(post_below_counts[i]), ha='center', va='top', fontsize=8)
        ax.text(i,  post_above_counts[i] + 0.5, str(post_above_counts[i]), ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Mask Area Ratio Bins')
    ax.set_xticks(index)
    ax.set_xticklabels([f'{mask_bins_post[i]:.2f}-{mask_bins_post[i+1]:.2f}' for i in range(n_bins)], rotation=45)

    # Shared legend on top
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.93])

    if savefig_name:
        outdir = f'.{env_vars["experiments_folder_path"]}/figs'
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


In [46]:
model_name = "polyp_pvt"
model_config = "b3"

best_ckpt_dir = "/mnt/c/Users/dewan/Desktop/oversampling_project_results/selected_best/"

pre_models = [m for m in os.listdir(best_ckpt_dir) if "pre_" in m]
post_models = [m for m in os.listdir(best_ckpt_dir) if "post_" in m]


In [48]:
area_vs_score_list_pre = get_area_vs_dice(data=test_data, model_name=model_name, model_config=model_config, ckpt_path=pre_models[0], device="cuda")

for model_ckpt in post_models:
    match = re.search(r'_(\d\.\d+)\.pt', model_ckpt)
    threshold_value = float(match.group(1))
    
    area_vs_score_list_post = get_area_vs_dice(
        data=test_data, 
        model_name=model_name, 
        model_config=model_config, 
        ckpt_path=f"{best_ckpt_dir}/{model_ckpt}", 
        device="cuda")

    plot_butterfly_mask_vs_score(
        n_bins=20, 
        threshold=threshold_value,
        pre_results=area_vs_score_list_pre,
        post_results=area_vs_score_list_post,
        savefig_name=f"area_vs_dice_at_{threshold_value}")